## 1. Setup and Configuration

In [ ]:
# Import required libraries
from databricks import sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Load environment variables from parent directory
load_dotenv(dotenv_path='../.env')

# Validate credentials
required_vars = ['DATABRICKS_SERVER_HOSTNAME', 'DATABRICKS_HTTP_PATH', 'DATABRICKS_TOKEN']
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

print("✓ Environment configured")

## 2. Analysis Parameters

In [ ]:
# Customer and time period settings
customer_filter = 'cds_8007'  # SAPPORO DRUG JP
start_date = '2023-01-01'
end_date = '2024-03-31'

# Product filters
category_filter = 'Laundry'
target_condition = "jp_sub_brand_alter_lang_name IN ('アリエール', 'ボールド')"  # Target products

# Trial/Repeat definition
lookback_days = 365  # Days to look back for previous purchases

print(f"✓ Parameters set")
print(f"  Analysis period: {start_date} to {end_date}")
print(f"  Category: {category_filter}")
print(f"  Target: {target_condition}")
print(f"  Lookback period: {lookback_days} days")

## 3. Build and Execute Query

In [ ]:
# Build trial & repeat classification query
query = f"""
WITH base_transactions AS (
    SELECT
        idpos.shopper_key AS shopper_id,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        jp_brand_alter_lang_name AS brand,
        jp_sub_brand_alter_lang_name AS sub_brand,
        jp_prod_name AS product,
        pos_unit_sales_qty AS unit,
        pos_sales_amt AS value,
        CASE WHEN {target_condition} THEN 1 ELSE 0 END AS is_target_product
    FROM
        cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
        LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
        LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE
        jp_category_name = '{category_filter}'
        AND idpos.data_provider_code_part = '{customer_filter}'
        AND sales_period_group_end_date_part BETWEEN '{start_date}' AND '{end_date}'
        AND shopper.member_ind = 'Y'
),
target_purchases AS (
    SELECT
        shopper_id,
        purchase_date,
        brand,
        sub_brand,
        product,
        unit,
        value,
        LAG(purchase_date, 1) OVER (PARTITION BY shopper_id ORDER BY purchase_date) AS prev_purchase_date,
        DATEDIFF(purchase_date, LAG(purchase_date, 1) OVER (PARTITION BY shopper_id ORDER BY purchase_date)) AS days_since_last
    FROM base_transactions
    WHERE is_target_product = 1
),
classified_purchases AS (
    SELECT
        shopper_id,
        purchase_date,
        brand,
        sub_brand,
        product,
        unit,
        value,
        prev_purchase_date,
        days_since_last,
        CASE
            WHEN prev_purchase_date IS NULL OR days_since_last > {lookback_days} THEN 'Trial'
            ELSE 'Repeat'
        END AS purchase_type,
        DATE_TRUNC('MONTH', purchase_date) AS purchase_month
    FROM target_purchases
)
SELECT
    shopper_id,
    purchase_date,
    purchase_month,
    brand,
    sub_brand,
    product,
    unit,
    value,
    purchase_type,
    days_since_last
FROM classified_purchases
ORDER BY purchase_date, shopper_id
"""

print("✓ SQL query constructed")
print(f"  Query length: {len(query)} characters")

In [ ]:
# Execute query
with sql.connect(
    server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path=os.getenv("DATABRICKS_HTTP_PATH"),
    access_token=os.getenv("DATABRICKS_TOKEN")
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        result = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        df = pd.DataFrame(result, columns=columns)

# Convert data types
df['purchase_date'] = pd.to_datetime(df['purchase_date'])
df['purchase_month'] = pd.to_datetime(df['purchase_month'])
df['unit'] = pd.to_numeric(df['unit'], errors='coerce')
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df['days_since_last'] = pd.to_numeric(df['days_since_last'], errors='coerce')

print(f"✓ Query executed successfully")
print(f"  Retrieved {len(df)} purchase transactions")
print(f"  Unique shoppers: {df['shopper_id'].nunique():,}")
print(f"\nFirst few rows:")
df.head(10)

## 4. Process and Analyze Results

In [ ]:
# Overall metrics
total_transactions = len(df)
total_shoppers = df['shopper_id'].nunique()
trial_transactions = len(df[df['purchase_type'] == 'Trial'])
repeat_transactions = len(df[df['purchase_type'] == 'Repeat'])
trial_shoppers = df[df['purchase_type'] == 'Trial']['shopper_id'].nunique()
repeat_shoppers = df[df['purchase_type'] == 'Repeat']['shopper_id'].nunique()

# Value metrics
total_value = df['value'].sum()
trial_value = df[df['purchase_type'] == 'Trial']['value'].sum()
repeat_value = df[df['purchase_type'] == 'Repeat']['value'].sum()

# Calculate conversion rate (shoppers who made both trial and repeat purchases)
trial_shoppers_set = set(df[df['purchase_type'] == 'Trial']['shopper_id'])
repeat_shoppers_set = set(df[df['purchase_type'] == 'Repeat']['shopper_id'])
converted_shoppers = len(trial_shoppers_set.intersection(repeat_shoppers_set))
conversion_rate = (converted_shoppers / len(trial_shoppers_set) * 100) if len(trial_shoppers_set) > 0 else 0

# Monthly trends
monthly_summary = df.groupby(['purchase_month', 'purchase_type']).agg({
    'shopper_id': 'nunique',
    'value': 'sum',
    'unit': 'sum'
}).reset_index()
monthly_summary.columns = ['month', 'type', 'shoppers', 'value', 'units']

print("=" * 60)
print("TRIAL & REPEAT ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nOverall Metrics:")
print(f"  Total Transactions: {total_transactions:,}")
print(f"  Total Shoppers: {total_shoppers:,}")
print(f"  Total Value: ¥{total_value:,.0f}")
print(f"\nTrial Purchases:")
print(f"  Transactions: {trial_transactions:,} ({trial_transactions/total_transactions*100:.1f}%)")
print(f"  Unique Shoppers: {trial_shoppers:,}")
print(f"  Value: ¥{trial_value:,.0f} ({trial_value/total_value*100:.1f}%)")
print(f"\nRepeat Purchases:")
print(f"  Transactions: {repeat_transactions:,} ({repeat_transactions/total_transactions*100:.1f}%)")
print(f"  Unique Shoppers: {repeat_shoppers:,}")
print(f"  Value: ¥{repeat_value:,.0f} ({repeat_value/total_value*100:.1f}%)")
print(f"\nConversion Metrics:")
print(f"  Trial shoppers who became repeat: {converted_shoppers:,}")
print(f"  Conversion rate: {conversion_rate:.1f}%")

# Average purchase interval for repeat buyers
avg_interval = df[df['purchase_type'] == 'Repeat']['days_since_last'].mean()
print(f"\nRepeat Purchase Behavior:")
print(f"  Average days between repeat purchases: {avg_interval:.1f}")

## 5. Visualizations

In [ ]:
# Pie chart: Trial vs Repeat transactions
type_summary = df.groupby('purchase_type').agg({
    'shopper_id': 'count',
    'value': 'sum'
}).reset_index()
type_summary.columns = ['Type', 'Transactions', 'Value']

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('By Transactions', 'By Value'),
    specs=[[{'type':'pie'}, {'type':'pie'}]]
)

colors = {'Trial': '#FF6B6B', 'Repeat': '#4ECDC4'}
color_list = [colors[t] for t in type_summary['Type']]

fig.add_trace(
    go.Pie(
        labels=type_summary['Type'],
        values=type_summary['Transactions'],
        marker_colors=color_list,
        texttemplate='%{label}<br>%{value:,}<br>(%{percent})',
        hole=0.3
    ),
    row=1, col=1
)

fig.add_trace(
    go.Pie(
        labels=type_summary['Type'],
        values=type_summary['Value'],
        marker_colors=color_list,
        texttemplate='%{label}<br>¥%{value:,.0f}<br>(%{percent})',
        hole=0.3
    ),
    row=1, col=2
)

fig.update_layout(
    title_text="Trial vs Repeat Distribution",
    height=400
)

fig.show()

In [ ]:
# Monthly trend: Trial vs Repeat shoppers
pivot_shoppers = monthly_summary.pivot(index='month', columns='type', values='shoppers').fillna(0)

fig = go.Figure()

if 'Trial' in pivot_shoppers.columns:
    fig.add_trace(go.Scatter(
        x=pivot_shoppers.index,
        y=pivot_shoppers['Trial'],
        mode='lines+markers',
        name='Trial',
        line=dict(color='#FF6B6B', width=3),
        marker=dict(size=8)
    ))

if 'Repeat' in pivot_shoppers.columns:
    fig.add_trace(go.Scatter(
        x=pivot_shoppers.index,
        y=pivot_shoppers['Repeat'],
        mode='lines+markers',
        name='Repeat',
        line=dict(color='#4ECDC4', width=3),
        marker=dict(size=8)
    ))

fig.update_layout(
    title='Trial vs Repeat Shoppers - Monthly Trend',
    xaxis_title='Month',
    yaxis_title='Number of Unique Shoppers',
    height=500,
    hovermode='x unified'
)

fig.show()

In [ ]:
# Stacked area chart: Monthly value contribution
pivot_value = monthly_summary.pivot(index='month', columns='type', values='value').fillna(0)

fig = go.Figure()

if 'Trial' in pivot_value.columns:
    fig.add_trace(go.Scatter(
        x=pivot_value.index,
        y=pivot_value['Trial'],
        mode='lines',
        name='Trial',
        fill='tonexty',
        line=dict(color='#FF6B6B', width=0),
        fillcolor='rgba(255, 107, 107, 0.5)'
    ))

if 'Repeat' in pivot_value.columns:
    fig.add_trace(go.Scatter(
        x=pivot_value.index,
        y=pivot_value['Repeat'],
        mode='lines',
        name='Repeat',
        fill='tonexty',
        line=dict(color='#4ECDC4', width=0),
        fillcolor='rgba(78, 205, 196, 0.5)'
    ))

fig.update_layout(
    title='Trial vs Repeat Sales Value - Monthly Trend',
    xaxis_title='Month',
    yaxis_title='Sales Value (¥)',
    height=500,
    hovermode='x unified'
)

fig.show()

In [ ]:
# Distribution of days between repeat purchases
repeat_intervals = df[df['purchase_type'] == 'Repeat']['days_since_last'].dropna()

fig = px.histogram(
    repeat_intervals,
    nbins=50,
    title='Distribution of Days Between Repeat Purchases',
    labels={'value': 'Days Since Last Purchase', 'count': 'Number of Purchases'},
    color_discrete_sequence=['#4ECDC4']
)

fig.add_vline(
    x=repeat_intervals.mean(),
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean: {repeat_intervals.mean():.0f} days",
    annotation_position="top"
)

fig.update_layout(height=400)
fig.show()

## 6. Data Tables

In [ ]:
# Monthly summary table
print("Monthly Trial & Repeat Summary:")
monthly_summary_display = monthly_summary.copy()
monthly_summary_display['month'] = monthly_summary_display['month'].dt.strftime('%Y-%m')
monthly_summary_display

In [ ]:
# Brand-level breakdown
brand_summary = df.groupby(['sub_brand', 'purchase_type']).agg({
    'shopper_id': 'nunique',
    'value': 'sum',
    'unit': 'sum'
}).reset_index()
brand_summary.columns = ['Sub-Brand', 'Type', 'Shoppers', 'Value', 'Units']
brand_summary = brand_summary.sort_values(['Sub-Brand', 'Shoppers'], ascending=[True, False])

print("\nBrand-Level Trial & Repeat Breakdown:")
brand_summary

## 7. Data Export

In [ ]:
# Export to Excel (optional)
export_file = f"trial_repeat_analysis_{category_filter}_{start_date}_to_{end_date}.xlsx"

with pd.ExcelWriter(export_file, engine='openpyxl') as writer:
    # Summary sheet
    summary_df = pd.DataFrame({
        'Metric': [
            'Total Transactions',
            'Total Shoppers',
            'Total Value',
            'Trial Transactions',
            'Trial Shoppers',
            'Trial Value',
            'Repeat Transactions',
            'Repeat Shoppers',
            'Repeat Value',
            'Converted Shoppers',
            'Conversion Rate %',
            'Avg Days Between Repeat'
        ],
        'Value': [
            f"{total_transactions:,}",
            f"{total_shoppers:,}",
            f"¥{total_value:,.0f}",
            f"{trial_transactions:,}",
            f"{trial_shoppers:,}",
            f"¥{trial_value:,.0f}",
            f"{repeat_transactions:,}",
            f"{repeat_shoppers:,}",
            f"¥{repeat_value:,.0f}",
            f"{converted_shoppers:,}",
            f"{conversion_rate:.1f}%",
            f"{avg_interval:.1f}"
        ]
    })
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Monthly trends
    monthly_summary.to_excel(writer, sheet_name='Monthly_Trends', index=False)
    
    # Brand breakdown
    brand_summary.to_excel(writer, sheet_name='Brand_Breakdown', index=False)
    
    # All transactions
    df.to_excel(writer, sheet_name='All_Transactions', index=False)

print(f"✓ Data exported to: {export_file}")